# GW in the atomic limit

In GW the self-energy is given by

$\Sigma(\nu) \approx \frac{Un}{2} -\frac{1}{\beta}\sum_{\nu^\prime}\left[\frac{1}{4}\eta^D(\nu-\nu^\prime)+\frac{3}{4}\eta^M(\nu-\nu^\prime)\right]$

with the bosonic propagators 

$\eta^{D,M}(\omega)=\frac{\pm U}{1\mp UP(\omega)}$

and the polarization bubble

$P(\omega) = \frac{1}{\beta}\sum_\nu G(\omega+\nu)G(\nu)$.

In [75]:
using MatsubaraFunctions
using NLsolve

As in the HF approximation, we define a function to compute the Matsubara sum over $G$:

In [76]:
function Matsubara_sum(f::MeshFunction)
    s = zero(eltype(G.data))
    νs = meshes(G,1)
    T  = temperature(νs)
    for i in eachindex(νs)
        s += f[i]
    end
    return T*s+0.5
end

Matsubara_sum (generic function with 1 method)

In [77]:
struct GWsolver
    T   :: Float64
    U   :: Float64
    N   :: Int64
    G   :: MeshFunction
    Σ   :: MeshFunction
    P   :: MeshFunction
    η_D :: MeshFunction
    η_M :: MeshFunction
    SGf :: SymmetryGroup
    SGb :: SymmetryGroup

    function GWsolver(T, U, N)

        # fermionic containers
        gf = MatsubaraMesh(T,N,Fermion)
        G  = MeshFunction(gf; data_t = ComplexF64)
        Σ  = MeshFunction(gf; data_t = ComplexF64)

        # bosonic containers
        gb  = MatsubaraMesh(T,N,Boson)
        P   = MeshFunction(ωs; data_t = ComplexF64)
        η_D = MeshFunction(ωs; data_t = ComplexF64)
        η_M = MeshFunction(ωs; data_t = ComplexF64)

        # symmetry groups
        conj_sym = Symmetry{1}() do args
            ω = args[1]
            ( -ω, ), Operation{ComplexF64}(sgn=false, con=true)
        end
        SGf = SymmetryGroup([conj_sym], G)
        SGb = SymmetryGroup([conj_sym], P)

        return new(T, U, N, G, Σ, P, η_D, η_M, SGf, SGb)
    end
end


In [124]:
function fixed_point!(F, x, S)

    # update Sigma
    unflatten!(S.Σ, x)

    # calculate G
    for i in eachindex(meshes(S.G,1))
        ν = value(value(points(meshes(S.G,1),i)))
        S.G[i] = 1.0 / (im * ν - S.Σ[i])
    end

    sum_mesh = MatsubaraMesh(S.T, 4*S.N, Fermion)

    # calculate P using symmetries
    calc_P = InitFunction{1, ComplexF64}(
        x -> begin
            ω = x[1]
            P = 0.0

            for i in eachindex(sum_mesh)
                ν = value(points(sum_mesh,i))
                P += S.G(ν + ω) * S.G(ν)
            end

            return S.T * P
        end
        )
    
    S.SGb(S.P, calc_P)

    # calculate η_D and η_M
    for i in meshes(S.η_D,1)
        S.η_D[i] = +S.U / (1.0 - S.U * S.P[i])
        S.η_M[i] = -S.U / (1.0 + S.U * S.P[i])
    end

    # calculate Σ using symmetries
    calc_Σ = InitFunction{1, ComplexF64}(
        x -> begin
            ν = x[1]
            Σ = S.U * Matsubara_sum(S.G)

            for i in eachindex(sum_mesh)
                νp = value(points(sum_mesh,i))
                Σ -= S.T * S.G(νp) * (
                    0.25 * S.η_D(ν - νp) +
                    0.75 * S.η_M(ν - νp) +
                    0.50 * S.U)
            end

            return Σ
        end
        )

    S.SGf(S.Σ, calc_Σ)

    # calculate the residue
    flatten!(S.Σ, F)
    F .-= x

    return nothing
end

fixed_point! (generic function with 1 method)

In [125]:
T = 0.3
U = 0.9
N = 1000

S = GWsolver(T, U, N)
init   = zeros(eltype(solver.G.data), length(S.G))
result = nlsolve((F,x) -> fixed_point!(F, x, S), init, method =:anderson, m = 8, beta = 0.5, show_trace = true)

Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     1     8.111336e-01              NaN
     2     3.110041e-01     8.836160e+00
     3     9.401311e-02     2.621184e-01
     4     4.118745e-02     9.012095e-02
     5     9.819527e-03     1.724087e-03
     6     6.969976e-04     4.982303e-05
     7     5.046918e-04     3.242705e-06
     8     6.651038e-06     4.285425e-09
     9     3.047073e-06     1.788227e-09
    10     1.068859e-08     1.056208e-14
    11     6.419606e-09     4.726257e-15


Results of Nonlinear Solver Algorithm
 * Algorithm: Anderson m=8 beta=0.5 aa_start=1 droptol=1.0e10
 * Starting Point: ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im,